# Schema Analysis

Basic analysis of mined RDF schemas from the LOD cloud.

In [1]:
import json
from collections import Counter
from pathlib import Path

import pandas as pd

OUTPUT_DIR = Path("/home/javier.millanacosta/rdfsolve/output")
print(f"Output directory: {OUTPUT_DIR}")

Output directory: /home/javier.millanacosta/rdfsolve/output


In [2]:
def load_schemas(output_dir: Path) -> dict:
    """Load all schema JSON-LD files."""
    schemas = {}
    for f in output_dir.glob("**/*_schema.jsonld"):
        name = f.parent.name
        try:
            data = json.loads(f.read_text())
            if data.get("@graph"):
                schemas[name] = data
        except Exception as e:
            print(f"Failed: {f.name}: {e}")
    return schemas

schemas = load_schemas(OUTPUT_DIR)
print(f"Loaded {len(schemas)} schemas")

Loaded 49 schemas


In [3]:
# Extract basic statistics
stats = []
for name, data in schemas.items():
    graph = data.get("@graph", [])
    classes = set()
    properties = set()
    patterns = 0
    
    for item in graph:
        if item.get("@id"):
            classes.add(item["@id"])
        for p in item.get("patterns", []):
            patterns += 1
            if p.get("property"):
                properties.add(p["property"])
    
    stats.append({
        "name": name,
        "classes": len(classes),
        "properties": len(properties),
        "patterns": patterns,
    })

df = pd.DataFrame(stats).sort_values("classes", ascending=False)
print(f"Total classes: {df['classes'].sum():,}")
print(f"Total properties: {df['properties'].sum():,}")
print(f"Total patterns: {df['patterns'].sum():,}")
df.head(20)

Total classes: 2,887,096
Total properties: 0
Total patterns: 0


,name,classes,properties,patterns
45,pubchem.ftp.substance,1607797,0,0
40,pubchem.ftp.protein,760522,0,0
33,lifesciencedict,490055,0,0
6,hra-kg,20420,0,0
34,wikipathways,1893,0,0
9,chembl,1208,0,0
48,nanosafetyrdf,668,0,0
1,glycosmos,540,0,0
10,uniprot,491,0,0
11,lslod_cloud,422,0,0


In [4]:
# Namespace distribution
def extract_namespace(uri: str) -> str:
    if "#" in uri:
        return uri.rsplit("#", 1)[0] + "#"
    elif "/" in uri:
        return uri.rsplit("/", 1)[0] + "/"
    return uri

namespace_counter = Counter()
for name, data in schemas.items():
    for item in data.get("@graph", []):
        if item.get("@id"):
            namespace_counter[extract_namespace(item["@id"])] += 1

print("Top 20 namespaces:")
for ns, count in namespace_counter.most_common(20):
    print(f"  {count:5d}  {ns[:60]}")

Top 20 namespaces:
  251019  http://purl.obolibrary.org/obo/
    547  https://identifiers.org/
     87  http://semanticscience.org/resource/
     48  http://www.bioassayontology.org/bao#
     47  https://rdfsolve.bigcat-bioinformatics.nl/dataset/
     45  partition:dt_4e598fd1b2c9
     24  dcterms:title
     20  partition:dt_0ebe96c73968
     16  dcterms:license
     15  dcterms:description
     15  http://edamontology.org/
     14  owl:Ontology
     14  partition:dt_13be789b5fff
     14  partition:dt_b8840ae319dd
     13  owl:ObjectProperty
     13  dcterms:creator
     12  skos:prefLabel
     12  owl:Class
     12  partition:dt_018c8a13e573
     12  http://purl.obolibrary.org/obo/chebi/


In [5]:
# Summary statistics
print("Schema size distribution:")
print(df[["classes", "properties", "patterns"]].describe())

Schema size distribution:
            classes  properties  patterns
count  4.900000e+01        49.0      49.0
mean   5.892033e+04         0.0       0.0
std    2.595619e+05         0.0       0.0
min    6.000000e+00         0.0       0.0
25%    2.200000e+01         0.0       0.0
50%    5.600000e+01         0.0       0.0
75%    3.600000e+02         0.0       0.0
max    1.607797e+06         0.0       0.0
